# 社区内容质检 · Qwen3Guard 多语言审核 · Colab GPU Demo

**目标**：用 Colab GPU 把开源内容安全模型 **Qwen3Guard-Gen-4B** 跑起来，
拿你社区里真实的多语言帖子验一验它的检出质量，判断能不能替代云盾这类商用审核服务。

**它是什么**：Qwen3Guard 是基于 Qwen3 训练的专职内容安全判别模型，
在 119 万条标注数据上训练，**支持 119 种语言**，多语言检出 F1 普遍 80-90%，
在开源 Guard 模型里是当前第一梯队（中文回复 F1 87.1%，同规格甩开 LlamaGuard3 约 20 个点）。

> ⚠️ **版本提示**：Qwen3Guard 基于 Qwen3 架构，**必须用较新版的 transformers**，
> 否则加载会报错。用一个干净的 Colab 会话跑本 notebook 即可（下面第 1 步会拉最新版）。

> 用法：菜单 **代码执行程序 → 更改运行时类型 → 硬件加速器选 T4 GPU**，然后从上往下逐格运行。

## 0. 确认拿到了 GPU

Qwen3Guard-Gen-4B 半精度约 8G 显存，T4（16G）放得下。看到 Tesla T4 就继续。

In [ ]:
!nvidia-smi

## 1. 装依赖

只需要 `transformers` + `accelerate`（分词器和推理都靠它俩）。

> **关键**：这里拉**最新版** transformers（`-U`），因为 Qwen3 架构需要较新的建模代码。
> 如果你不小心复用了翻译 notebook 那个降级过的运行时，这一步会把它升回来 ——
> 但更稳妥的做法还是**新开一个会话**。

In [ ]:
!pip install -q -U transformers accelerate

## 2. 加载 Qwen3Guard-Gen-4B 到 GPU

`Gen` 版本把"安全分类"当成一个指令跟随任务：喂它一段内容，它生成一段结构化判定文本。
（另有 `Stream` 版本用于边生成边实时拦截，那是给 LLM 输出做护栏用的，我们审帖子用 Gen 版。）

In [ ]:
# ===== 下载卡死排查格（正常情况跳过，卡 0% 时才用）=====
# ① 看缓存目录有没有字节落盘：反复跑这行，数字在涨 = 进度条假死但其实在下，等它就行
!du -sh /root/.cache/huggingface 2>/dev/null || echo '缓存目录还不存在'

# ② 测 Colab 到 HF 的裸连通性：应该秒回 200，速度是 B/s
!curl -s -o /dev/null -w 'HTTP %{http_code}  下载速度 %{speed_download} B/s\n' -L \
  https://huggingface.co/Qwen/Qwen3Guard-Gen-4B/resolve/main/config.json

In [ ]:
# ===== 兜底方案：用命令行工具下载（有真实进度条，下完 from_pretrained 直接命中缓存）=====
# 上面②连通性正常、但 python 里死活卡 0% 时用这格。约 8G，正常几分钟。
!HF_HUB_DISABLE_XET=1 hf download Qwen/Qwen3Guard-Gen-4B

In [6]:
import os
# Colab 上 HF 的 Xet 下载后端偶发卡死（一直 0% 不报错），禁用它退回普通 HTTP 下载
# 必须在 import transformers 之前设置才生效
os.environ['HF_HUB_DISABLE_XET'] = '1'

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'Qwen/Qwen3Guard-Gen-4B'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype='auto',   # T4 上会用 fp16，约 8G 显存
    device_map='auto',
)
# 显存紧张时（比如同卡还挂着别的模型）可改成 4bit，需先 pip install bitsandbytes：
#   model = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit=True, device_map='auto')
print('审核模型就绪 ✅')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.60k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

审核模型就绪 ✅


## 3. 定义审核函数（调用 + 解析）

Qwen3Guard-Gen 的输出是一段结构化文本，形如：

```
Safety: Unsafe
Categories: Suicide & Self-Harm
```

我们用官方给的正则把它解析成 `{label, categories}`。

**三级严重度**（这是它比普通"违规/不违规"二分类高级的地方）：
- **Safe** —— 绝大多数场景都安全
- **Unsafe** —— 绝大多数场景都有害
- **Controversial** —— 有害与否要看上下文（灰色地带，最适合丢人工）

**九大类别**：Violent（暴力）、Non-violent Illegal Acts（非暴力违法）、
Sexual Content（色情）、PII（个人隐私信息）、Suicide & Self-Harm（自伤）、
Unethical Acts（不道德行为）、Politically Sensitive Topics（涉政敏感）、
Copyright Violation（侵权）、Jailbreak（越狱攻击）。

In [7]:
import re

def parse(content: str):
    '''从模型输出里抠出 安全标签 + 违规类别。正则来自官方 model card。'''
    label_m = re.search(r'Safety: (Safe|Unsafe|Controversial)', content)
    label = label_m.group(1) if label_m else None
    cats = re.findall(
        r'(Violent|Non-violent Illegal Acts|Sexual Content or Sexual Acts|PII|'
        r'Suicide & Self-Harm|Unethical Acts|Politically Sensitive Topics|'
        r'Copyright Violation|Jailbreak|None)', content)
    return label, cats

def moderate(text: str, role: str = 'user') -> dict:
    '''审一段内容。role=user 表示审用户发布的帖子（Qwen3Guard 按对话角色套审核模板）。'''
    messages = [{'role': role, 'content': text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        # do_sample=False：审核要可复现，关掉随机性
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    gen_ids = out[0][len(inputs.input_ids[0]):]
    content = tokenizer.decode(gen_ids, skip_special_tokens=True)
    label, cats = parse(content)
    return {'label': label, 'categories': cats, 'raw': content}

print('测一条：', moderate('大家好，欢迎加入我们的社区！'))

测一条： {'label': 'Safe', 'categories': ['None'], 'raw': 'Safety: Safe\nCategories: None'}


## 4. 审你自己的真实社区文本 ⭐（多语言）

**把下面的 `samples` 换成你社区里真实的帖子**（各语种、各类型都来），
这是整个 notebook 最关键的一步 —— 检出质量只有用你的真实数据才算数。

下面这几条是**故意设计的教学样本**，覆盖"正常 / 明显违规 / 灰色 / 分类学盖不到"四种情况，
每条我都标了**预期**，跑完对一对模型判得准不准。

In [9]:
# (文本, 说明, 预期) —— 换成你的真实数据
samples = [
    ('社区今晚 8 点有新版本发布会，欢迎大家来看直播~',
     '正常活动通知', '预期 Safe'),

    ('You are all worthless idiots, this community is trash and so are you.',
     '英文辱骂', '预期 Unsafe / Unethical Acts'),

    ('最近真的撑不下去了，感觉活着没什么意义，想结束这一切。',
     '中文自伤倾向', '预期 Unsafe / Suicide & Self-Harm —— 审核系统必须接住'),

    ('加我微信 abc123 领取内部福利，站内消息别回，直接加好友。',
     '广告导流', '预期可能被判 Safe ← 教学点：导流不在"安全分类学"里，Guard 模型天生不管，得靠词表层补'),
]

for text, note, expect in samples:
    r = moderate(text)
    print(f'【{note}】{expect}')
    print(f'  原文：{text}')
    print(f'  判定：{r["label"]}  类别：{r["categories"]}')
    print()

【正常活动通知】预期 Safe
  原文：社区今晚 8 点有新版本发布会，欢迎大家来看直播~
  判定：Safe  类别：['None']

【英文辱骂】预期 Unsafe / Unethical Acts
  原文：You are all worthless idiots, this community is trash and so are you.
  判定：Controversial  类别：['Unethical Acts']

【中文自伤倾向】预期 Unsafe / Suicide & Self-Harm —— 审核系统必须接住
  原文：最近真的撑不下去了，感觉活着没什么意义，想结束这一切。
  判定：Unsafe  类别：['Suicide & Self-Harm']

【广告导流】预期可能被判 Safe ← 教学点：导流不在"安全分类学"里，Guard 模型天生不管，得靠词表层补
  原文：加我微信 abc123 领取内部福利，站内消息别回，直接加好友。
  判定：Controversial  类别：['Non-violent Illegal Acts']



## 5. 三级严重度 → 三条业务处置路线

审核模型的产出不是终点，**怎么根据判定分流处置**才是业务。
这里演示一个负责任的处置策略：

- **Safe** → ✅ 放行，进翻译流水线
- **Unsafe** → 按类别细分：一般违规直接拦截；**自伤类特殊处理**（不是简单删帖，
  而是触发关怀/求助资源推送 —— 这是内容安全最有价值的正面用途之一）
- **Controversial** → ⚠️ 进人工审核队列，人判完的结果回流成标注数据

**对照第 4 步的实测结果，两条和"预期"有出入，恰好都是教学点**：

- **英文辱骂被判 Controversial 而不是 Unsafe** —— 纯辱骂（无威胁、无煽动）在 Qwen3Guard
  的分类学里就是"看语境定性"的灰色地带：竞技社区的垃圾话和恶意人身攻击措辞可能一模一样。
  这不是漏判，而是三级严重度**把定性权交给你的社区规则** —— 你的社区如果零容忍，
  就在处置层把 `Controversial + Unethical Acts` 直接升级成拦截，而不是指望模型替你收紧。
- **"加微信导流"没有漏，被判 Controversial / Non-violent Illegal Acts** —— 模型抓住的是
  "内部福利、别走站内、私下加好友"这套**疑似诈骗话术**，而不是"导流"本身（分类学里没有导流）。
  换句话说：写得像骗局的导流它能接住，**写得人畜无害的广告导流依然会漏**。
  所以生产系统里前面还是得挂一层**词表/正则规则**（扫微信号、外链等导流特征），两层配合才够用。


In [11]:
def handle(text: str) -> str:
    '''根据审核判定给出处置动作。返回处置说明字符串。'''
    r = moderate(text)
    label, cats = r['label'], r['categories']

    if label == 'Safe':
        return '✅ 放行 → 进翻译流水线'
    if label == 'Controversial':
        return f'⚠️ 进人工审核队列（灰色，类别参考：{cats}）'
    # Unsafe：按类别细分处置
    if 'Suicide & Self-Harm' in cats:
        return '🆘 拦截并触发关怀流程：推送心理求助热线/资源，通知社区管理员（不做冷冰冰的删帖）'
    return f'⛔ 拦截（违规类别：{cats}）'

for text, note, _ in samples:
    print(f'【{note}】')
    print(f'  → {handle(text)}')
    print()

【正常活动通知】
  → ✅ 放行 → 进翻译流水线

【英文辱骂】
  → ⚠️ 进人工审核队列（灰色，类别参考：['Unethical Acts']）

【中文自伤倾向】
  → 🆘 拦截并触发关怀流程：推送心理求助热线/资源，通知社区管理员（不做冷冰冰的删帖）

【广告导流】
  → ⚠️ 进人工审核队列（灰色，类别参考：['Non-violent Illegal Acts']）



## 6. 量一下速度

审核是"输入长、输出短"（判定文本才十几个 token），所以比翻译快得多。
跑几条测平均延迟，感受一下同一张卡上审核这道工序的开销。

In [ ]:
import time

moderate('热身一下')  # 预热
t = time.perf_counter()
N = 10
for _ in range(N):
    moderate('社区今晚有一场新版本直播，欢迎大家来看。')
dt = (time.perf_counter() - t) / N
print(f'单条审核平均延迟：{dt*1000:.0f} ms')

### 6.5 跑量吞吐：批量推理

上面测的是**延迟**（一条来了多久出结果），跑量看的是**吞吐**（一秒能出多少条），两者不是一回事。

batch=1 时 GPU 大部分算力在闲着：生成每个 token 的瓶颈是把 8G 权重从显存搬进计算单元，
搬一遍权重只服务一条文本太亏了 —— **同一遍搬运顺便算 32 条，几乎不多花时间**。
所以把待审文本拼成一批一起喂，单条折合成本会大幅下降。跑量（比如夜间批审历史帖）都该这么干。

In [16]:
import time
def moderate_batch(texts: list[str]) -> list[dict]:
    '''批量审核：一次 generate 同时处理整批文本，吞吐远高于逐条调用。'''
    # decoder 模型批量生成必须左填充（让所有序列的"末尾"对齐，模型才知道从哪续写）
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    prompts = [tokenizer.apply_chat_template([{'role': 'user', 'content': t}],
                                             tokenize=False, add_generation_prompt=True)
               for t in texts]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    results = []
    for i in range(len(texts)):
        gen_ids = out[i][inputs.input_ids.shape[1]:]
        content = tokenizer.decode(gen_ids, skip_special_tokens=True)
        label, cats = parse(content)
        results.append({'label': label, 'categories': cats})
    return results

# 拿同一条文本复制 32 份压测（真实使用时换成 32 条不同的帖子）
BATCH = 32
texts = ['社区今晚有一场新版本直播，欢迎大家来看。'] * BATCH

moderate_batch(texts[:2])  # 预热
t = time.perf_counter()
res = moderate_batch(texts)
dt = time.perf_counter() - t
print(f'批量 {BATCH} 条总耗时 {dt:.1f} s')
print(f'折合单条 {dt/BATCH*1000:.0f} ms（对比逐条的 1825 ms）')
print(f'吞吐 {BATCH/dt:.1f} 条/秒 ≈ {BATCH/dt*3600:.0f} 条/小时')


批量 32 条总耗时 35.0 s
折合单条 1094 ms（对比逐条的 1825 ms）
吞吐 0.9 条/秒 ≈ 3289 条/小时


## 7. 小结 & 生产落地

**你在这个 notebook 里验证了**：加载 Qwen3Guard → 审多语言帖子 → 三级严重度分流处置 → 压测，
一整条内容质检的核心链路。

**从 demo 到生产，四层架构**（把云盾的"运营质量"自己搭起来）：

```
第1层 词表/正则规则     扫违禁词、导流特征（微信号/外链）  ← 改了即生效，补 Guard 盖不到的
第2层 Qwen3Guard 判别   多语言语义检出，高置信度直接判      ← 本 notebook 这层
第3层 大模型仲裁        Guard 拿不准的丢给同卡的 Qwen3-30B  ← 疑难件二审，边际成本≈0
第4层 人工队列 + 回流    三审仍不准的人工判，判例落库        ← 喂词表(天级) + 微调数据(季度级)
```

**低频微调的够用节奏**：
- 具体新违禁词 → 进**第 1 层词表**，即时生效，不劳驾微调
- 只能靠"看例子才懂"的新语义模式 → 攒够 **2000~5000 条判例**再 LoRA 微调一次（约季度节奏）
- 微调前留 500 条历史判例做**回归集**，新版不低于旧版才上线；数据里混 10-20% 原始安全样本防遗忘

**部署形态**：和翻译一样用 vLLM 起 OpenAI 兼容服务，审核模型 + 翻译模型同卡共存。
